# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

changed order 3 to V-08 to create a missing vendor.

**One row of my result is one:** line item.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
import pandas as pd
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

assert len(lines) == 4
assert lines['qty'].sum() == 7
print(lines)

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-08


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
vendors = pd.DataFrame(vendors_json)
joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)
print(joined)

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-08           NaN  NaN  left_only


In [ ]:
print(joined['_merge'].value_counts())
print(joined[joined['_merge'] == 'left_only'])

_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64
      item  qty order vendor_id name zone     _merge
3  Hot Dog    3     3      V-08  NaN  NaN  left_only


V-08 didn't match and it has 3 units attached to it.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
vendor_totals = (
    joined
    .groupby('name', dropna=False)['qty']
    .sum()
    .reset_index()
)

print(vendor_totals)

           name  qty
0     Cav Merch    1
1  Hoos Burgers    3
2           NaN    3


It appears as 3 NaN units, this is because dropna=False.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
zone_totals = (
    joined
    .groupby('zone', dropna=False)['qty']
    .sum()
    .reset_index()
)

print(zone_totals)

  zone  qty
0    A    4
1  NaN    3


Zone A sold the most with 4 units. The total is not 7 because 3 units are in the unknown vendor so it doesn't have a zone.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

ragged_with_lines = [
    order for order in ragged_json
    if 'lines' in order
]

ragged_lines = pd.json_normalize(
    ragged_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(ragged_lines)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'

print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A) One row per line item is usefk because we can look at individual items in each order. We know easily how many units are in each vendor and zone. If we had one row per order than it would be hard to differentiate different items because it would group those items together.

B) A qauntity of 0 means we know no units were sold, and a missing quantity means we don't know how many units were sold. If we change all of the missing quantities to 0 then we would be acting like we know the missing information is 0. The totals would be inaccurate.